In [213]:
import requests
from bs4 import BeautifulSoup
import json
import re

url = "https://sweb.hku.hk/ccacad/ccc_appl/enrol_stat.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

enrol_data = {}
current_sem = "sem1"

for elem in soup.find_all(["h3", "h4", "p", "b", "td", "tr"]):
    text = elem.get_text().strip()
    if "Second Semester" in text and elem.name != "tr":
        current_sem = "sem2"
    elif "First Semester" in text and elem.name != "tr":
        current_sem = "sem1"

    if elem.name == "tr":
        tds = [td.get_text().strip() for td in elem.find_all("td")]
        if len(tds) >= 6 and re.match(r"^CC[A-Z]{2}\d{4}$", tds[0]):
            code, title, subclass, quota, vacancies, waiting = tds[:6]
            code = code.strip().upper()
            subclass = subclass.strip().upper()

            # Skip CCAI and skip Exchange subclasses (AX, BX, CX, DX)
            if code.startswith("CCAI") or subclass.endswith("X"):
                continue

            try:
                quota = int(quota)
                vacancies = int(vacancies)
            except ValueError:
                continue

            if code not in enrol_data:
                enrol_data[code] = {
                    "sem1_quota": 0, "sem1_vacancies": 0,
                    "sem2_quota": 0, "sem2_vacancies": 0,
                    "regular_subclasses": []
                }

            enrol_data[code][f"{current_sem}_quota"] += quota
            enrol_data[code][f"{current_sem}_vacancies"] += vacancies
            enrol_data[code]["regular_subclasses"].append(f"{current_sem}_{subclass}")

with open("enrol-stat-info.json", "w") as f:
    json.dump(enrol_data, f, indent=4)

print(f"Saved clean regular-student enrollment stats for {len(enrol_data)} non-CCAI courses to 'enrol-stat-info.json'!")

Saved clean regular-student enrollment stats for 171 non-CCAI courses to 'enrol-stat-info.json'!


In [ ]:
!pip install -q openai

In [ ]:
import json
import re
import time
from openai import OpenAI

# 1. Paste your Poe API key here:
POE_API_KEY = "sk-poe-cpdaq2F_PgXw-ZyjCgOqUaECDTEEfM6xk6VeytxanB4" #api key is deleted so its fine to expose now

client = OpenAI(
    api_key=POE_API_KEY,
    base_url="https://api.poe.com/v1"
)

# 2. Hardcoded exact model ID from your Poe output!
MODEL_NAME = "deepseek-v4.1-flash-1017"
print(f"Using exact Poe Model ID: {MODEL_NAME}")

# 3. Load richku data (supports either richku-info.json or ric-info.json)
try:
    with open("richku-info.json", "r") as f:
        ric_data = json.load(f)
except FileNotFoundError:
    with open("ric-info.json", "r") as f:
        ric_data = json.load(f)

# Load anchor courses (170 active non-CCAI courses from cc-site-info.json)
with open("cc-site-info.json", "r") as f:
    cc_data = json.load(f)
active_codes = sorted([c.strip().upper() for c in cc_data.keys() if not c.strip().upper().startswith("CCAI")])

# Collect reviews ONLY for active courses that actually have reviews
courses_with_reviews = {}
for code in active_codes:
    c_data = ric_data.get(code, {})
    reviews = c_data.get("reviews", [])
    if reviews:
        texts = []
        for r in reviews:
            txt = (r.get("translated_content") or r.get("content") or "").strip()
            if txt:
                texts.append(txt)
        if texts:
            courses_with_reviews[code] = texts

print(f"Active non-CCAI courses with text reviews on RIC: {len(courses_with_reviews)} / {len(active_codes)}")

# Reset llm_scores to start fresh with deepseek-v4.1-flash-1017
llm_scores = {}

# 4. Run 1 API call per course
codes_to_score = list(courses_with_reviews.keys())

for idx, code in enumerate(codes_to_score):
    reviews_list = courses_with_reviews[code]
    formatted_reviews = "\n".join([f"- Review {i+1}: {rev}" for i, rev in enumerate(reviews_list)])

    prompt = f"""You are evaluating student reviews for the HKU Common Core course {code}.
Read the following {len(reviews_list)} student reviews and output two continuous scores (from 1.0 to 5.0, rounded to 2 decimal places):
1. "llm_sentiment_score": Overall student experience/sentiment from 1.0 (terrible experience, unfair grading, boring) to 5.0 (amazing experience, interesting, great teaching/grading).
2. "llm_workload_score": Perceived course workload from 1.0 (very chill, light workload, easy) to 5.0 (very heavy workload, stressful, demanding).

Student Reviews for {code}:
{formatted_reviews}

Respond ONLY with a valid JSON object in this exact format, with no extra text or markdown:
{{"llm_sentiment_score": 3.50, "llm_workload_score": 2.80}}"""

    print(f"[{idx+1}/{len(codes_to_score)}] Scoring {code} ({len(reviews_list)} reviews)...", end=" ")

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You are a precise data science assistant that outputs only valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0
            )
            raw_reply = response.choices[0].message.content.strip()

            json_match = re.search(r"\{[^{}]*\"llm_sentiment_score\"[^{}]*\}", raw_reply, re.DOTALL)
            if not json_match:
                raise ValueError(f"Could not find JSON in response: {raw_reply[:100]}")

            parsed = json.loads(json_match.group(0))
            sent_score = round(float(parsed["llm_sentiment_score"]), 2)
            work_score = round(float(parsed["llm_workload_score"]), 2)

            sent_score = max(1.0, min(5.0, sent_score))
            work_score = max(1.0, min(5.0, work_score))

            llm_scores[code] = {
                "llm_sentiment_score": sent_score,
                "llm_workload_score": work_score,
                "ric_review_count": len(reviews_list)
            }
            print(f"OK -> Sentiment: {sent_score}, Workload: {work_score}")
            break

        except Exception as e:
            print(f"\n  Attempt {attempt+1} error on {code}: {e}")
            time.sleep(2)

    if (idx + 1) % 10 == 0:
        with open("ric_llm_scores.json", "w") as f:
            json.dump(llm_scores, f, indent=4)

with open("ric_llm_scores.json", "w") as f:
    json.dump(llm_scores, f, indent=4)

print(f"\nDone! Scored {len(llm_scores)} courses with {MODEL_NAME} and saved to 'ric_llm_scores.json'!")

In [214]:
import json
import pandas as pd
import numpy as np
import re

# 1. Load all JSON files
with open("cc-site-info.json", "r") as f:
    cc_raw = json.load(f)

with open("sis-info.json", "r") as f:
    sis_raw = json.load(f)

with open("enrol-stat-info.json", "r") as f:
    enrol_raw = json.load(f)

try:
    with open("ric_llm_scores.json", "r") as f:
        llm_raw = json.load(f)
except FileNotFoundError:
    llm_raw = {}

# ---------------------------------------------------------
# 2. Rule-Based Categorization Functions
# ---------------------------------------------------------
unmatched_load_keys = set()
unmatched_ass_keys = set()

def categorize_study_load(load_dict):
    """
    Maps study_load_dist into:
    hours_lectures, hours_tutorials, hours_self_study, hours_assessment
    """
    buckets = {
        "hours_lectures": 0.0,
        "hours_tutorials": 0.0,
        "hours_self_study": 0.0,
        "hours_assessment": 0.0
    }
    for raw_k, raw_v in load_dict.items():
        k = raw_k.strip().lower()
        val = float(str(raw_v).strip())

        # Rule 1: Anything starting with "assessment:" or mentioning quiz/test/exam/essay/report/project/presentation/assignment/paper
        if k.startswith("assessment") or any(w in k for w in [
            "essay", "report", "presentation", "project", "quiz", "test", "exam",
            "assignment", "writing", "journal", "paper", "portfolio", "poster",
            "debate", "podcast", "video", "blog", "critique", "reflection", "preparation for"
        ]):
            buckets["hours_assessment"] += val
        # Rule 2: Lectures / Seminars
        elif any(w in k for w in ["lecture", "seminar"]):
            buckets["hours_lectures"] += val
        # Rule 3: Reading / Self-study
        elif any(w in k for w in ["reading", "self-study", "self study", "online learning", "independent study", "revision"]):
            buckets["hours_self_study"] += val
        # Rule 4: Interactive / Experiential classes (Tutorials, Fieldwork, Visits, Labs, Workshops, Film viewing)
        elif any(w in k for w in [
            "tutorial", "field", "visit", "trip", "workshop", "lab", "film",
            "screening", "discussion", "practicum", "studio", "excursion", "activity", "consultation"
        ]):
            buckets["hours_tutorials"] += val
        else:
            unmatched_load_keys.add(raw_k)
            buckets["hours_self_study"] += val # Default fallback

    return buckets

def categorize_assessment(ass_dict):
    """
    Maps ass_dist into:
    weight_essay, weight_quiz_exam, weight_presentation, weight_participation, weight_project
    If a compound task matches multiple categories (e.g. 'Group project and presentation'),
    splits the weight evenly across the matched categories!
    """
    buckets = {
        "weight_essay": 0.0,
        "weight_quiz_exam": 0.0,
        "weight_presentation": 0.0,
        "weight_participation": 0.0,
        "weight_project": 0.0
    }

    for raw_k, raw_v in ass_dict.items():
        k = raw_k.strip().lower()
        val = float(str(raw_v).strip().replace("%", "").strip())

        matched_cats = []

        # Check Essay / Written work
        if any(w in k for w in [
            "essay", "writing", "written", "report", "journal", "paper",
            "reflection", "reflective", "review", "critique", "commentary",
            "diary", "blog", "response", "summary", "editorial", "opinion", "article"
        ]):
            matched_cats.append("weight_essay")

        # Check Quiz / Test / Exam
        if any(w in k for w in [
            "quiz", "test", "exam", "mcq", "multiple choice", "multiple-choice", "mid-term", "midterm"
        ]) and not any(w in k for w in ["essay", "paper", "report", "project"]):
            matched_cats.append("weight_quiz_exam")
        elif any(w in k for w in ["quiz", "test", "exam", "mcq"]):
            matched_cats.append("weight_quiz_exam")

        # Check Presentation / Oral / Debate
        if any(w in k for w in [
            "presentation", "oral", "debate", "pitch", "speech", "symposium", "panel", "role-play", "role play"
        ]):
            matched_cats.append("weight_presentation")

        # Check Participation / Attendance / In-class / Tutorial exercises
        if any(w in k for w in [
            "participation", "attendance", "discussion", "in-class", "in class",
            "tutorial exercise", "tutorial task", "tutorial work", "tutorial activity",
            "classwork", "class exercise", "engagement", "forum", "peer", "homework", "worksheet"
        ]):
            matched_cats.append("weight_participation")

        # Check Project / Creative / Poster / Podcast / Video / Portfolio / Assignment
        if any(w in k for w in [
            "project", "poster", "podcast", "video", "multimedia", "portfolio",
            "exhibition", "creative", "design", "prototype", "fieldwork", "case study", "dossier"
        ]):
            matched_cats.append("weight_project")

        # Fallback for generic words like "Assignments", "Continuous assessment", "Tutorials"
        if not matched_cats:
            if "tutorial" in k or "lecture" in k:
                matched_cats.append("weight_participation")
            elif "assignment" in k or "work" in k or "task" in k:
                matched_cats.append("weight_essay")
            elif "assessment" in k:
                matched_cats.append("weight_quiz_exam")
            else:
                unmatched_ass_keys.add(raw_k)
                matched_cats.append("weight_project")

        # Split weight evenly across all matched categories for this item
        split_val = val / len(matched_cats)
        for cat in matched_cats:
            buckets[cat] += split_val

    # Normalize so the 5 weights strictly sum to 100.0%
    total_w = sum(buckets.values())
    if total_w > 0:
        for cat in buckets:
            buckets[cat] = round((buckets[cat] / total_w) * 100.0, 2)

    return buckets

# ---------------------------------------------------------
# 3. Pre-aggregate SIS Dataset (Response-Weighted Average)
# ---------------------------------------------------------
sis_clean = {}
for code, records in sis_raw.items():
    code = code.strip().upper()
    if code.startswith("CCAI"):
        continue
    valid_records = []
    for r in records:
        try:
            resp = float(r.get("responses", 0))
            eff = float(r.get("mean_course_effective", np.nan))
            if resp > 0 and not np.isnan(eff):
                valid_records.append((resp, eff))
        except ValueError:
            continue
    if valid_records:
        tot_resp = sum(x[0] for x in valid_records)
        weighted_eff = sum(x[0] * x[1] for x in valid_records) / tot_resp
        sis_clean[code] = round(weighted_eff, 2)

# ---------------------------------------------------------
# 4. Build Main Table Anchored on cc-site-info.json (Non-CCAI)
# ---------------------------------------------------------

print(f"len(cc_raw)={len(cc_raw)}")

rows = []
for code, info in sorted(cc_raw.items()):
    code = code.strip().upper()
    if code.startswith("CCAI"):
        continue

    course_name = info.get("course_name", "").strip()
    aoi = code[:4] # CCST, CCHU, CCGL, CCCH

    # Check if Online course (either 5000-level code or "(Online)" in course name)
    is_online = 1 if ("online" in course_name.lower() or code[4] == "5") else 0

    # Check if Communication-intensive
    is_comm_intensive = int(info.get("is_comm_intensive") is True)

    # Parse Semester Offered ("sem1", "sem2", or "both")
    sem_str = info.get("sem_offered", "").lower()
    has_sem1 = "first" in sem_str or "sem 1" in sem_str
    has_sem2 = "second" in sem_str or "sem 2" in sem_str
    if has_sem1 and has_sem2:
        offered_sem = "both"
    elif has_sem1:
        offered_sem = "sem1"
    elif has_sem2:
        offered_sem = "sem2"
    else:
        offered_sem = "unknown"

    # Tally SDGs
    sdg_buckets = {f"sdg_{i:02d}": 0 for i in range(1, 18)}
    for sdg in info.get("sdgs"):
        sdg_buckets[f"sdg_{int(sdg[-2:]):02d}"] = 1

    # Study Load Features
    study_load_total = float(str(info.get("study_load_total", "0")).strip())
    load_buckets = categorize_study_load(info.get("study_load_dist", {}))

    # Assessment Weight Features (Normalized to 100%)
    ass_buckets = categorize_assessment(info.get("ass_dist", {}))

    # Enrolment Features (Quota per semester & Fill rate)
    enrol_info = enrol_raw.get(code, {})
    s1_q = enrol_info.get("sem1_quota", 0)
    s1_v = enrol_info.get("sem1_vacancies", 0)
    s2_q = enrol_info.get("sem2_quota", 0)
    s2_v = enrol_info.get("sem2_vacancies", 0)

    num_sems_active = (1 if s1_q > 0 else 0) + (1 if s2_q > 0 else 0)
    total_q = s1_q + s2_q
    total_v = s1_v + s2_v

    quota_per_semester = (total_q / num_sems_active) if num_sems_active > 0 else np.nan
    enrol_fill_rate = round((total_q - total_v) / total_q, 4) if total_q > 0 else np.nan
    sem1_fill_rate = round((s1_q - s1_v) / s1_q, 4) if s1_q > 0 else np.nan
    sem2_fill_rate = round((s2_q - s2_v) / s2_q, 4) if s2_q > 0 else np.nan

    # Target Metrics (Y): SIS & LLM Scores
    sis_mean_effectiveness = sis_clean.get(code, np.nan)
    llm_info = llm_raw.get(code, {})
    llm_sentiment_score = llm_info.get("llm_sentiment_score", np.nan)
    llm_perceived_workload = llm_info.get("llm_workload_score", np.nan)

    rows.append({
        "course_code": code,
        "course_name": course_name,
        # --- Course Profile Features (X) ---
        "aoi": aoi,
        "is_online": is_online,
        "is_comm_intensive": is_comm_intensive,
        "offered_sem": offered_sem,
        **sdg_buckets,
        "study_load_total": study_load_total,
        **load_buckets,
        **ass_buckets,
        "quota_per_semester": quota_per_semester,
        # --- Course Performance Metrics (Y) ---
        "sis_mean_effectiveness": sis_mean_effectiveness,
        "enrol_fill_rate": enrol_fill_rate,
        "sem1_fill_rate": sem1_fill_rate,
        "sem2_fill_rate": sem2_fill_rate,
        "llm_sentiment_score": llm_sentiment_score,
        "llm_perceived_workload": llm_perceived_workload
    })

df_main = pd.DataFrame(rows)

# One-Hot Encode 'aoi' and 'offered_sem' (keeping original columns too for EDA)
df_main = pd.get_dummies(df_main, columns=["aoi", "offered_sem"], prefix=["aoi", "sem"], dtype=int, drop_first=False)
# Re-add string 'aoi' and 'offered_sem' at the front for convenient EDA plotting
df_main.insert(2, "aoi", df_main["course_code"].str[:4])

# Add Semester-Adjusted Popularity Percentile (solves Sem 1 vs Sem 2 vacancy imbalance!)
s1_pct = df_main["sem1_fill_rate"].rank(pct=True)
s2_pct = df_main["sem2_fill_rate"].rank(pct=True)
df_main["enrol_popularity_pct_sem_adj"] = np.nanmean(np.vstack([s1_pct, s2_pct]), axis=0).round(4)

print(f"Unmatched study_load keys ({len(unmatched_load_keys)}):", unmatched_load_keys)
print(f"Unmatched ass_dist keys ({len(unmatched_ass_keys)}):", unmatched_ass_keys)
print("\n--- MISSING DATA SUMMARY (BEFORE MCAR IMPUTATION) ---")
print(df_main[["quota_per_semester", "sis_mean_effectiveness", "enrol_fill_rate", "llm_sentiment_score", "llm_perceived_workload"]].isnull().sum())

# Save 1: Dataset with NaNs preserved (for EDA & checking distributions)
df_main.to_csv("hku_cc_main_dataset_raw_na.csv", index=False)

# Save 2: Dataset with MCAR Median Imputation (for Modelling)
df_imputed = df_main.copy()
cols_to_impute = ["quota_per_semester", "sis_mean_effectiveness", "enrol_fill_rate", "enrol_popularity_pct_sem_adj", "llm_sentiment_score", "llm_perceived_workload"]
for col in cols_to_impute:
    if df_imputed[col].notnull().any():
        df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

df_imputed.to_csv("hku_cc_main_dataset_imputed.csv", index=False)
print("\nSaved both 'hku_cc_main_dataset_raw_na.csv' and 'hku_cc_main_dataset_imputed.csv'!")
df_main.head()

len(cc_raw)=270
Unmatched study_load keys (20): {'Group work', 'Demonstrations', 'Online course materials', 'Independent research and fact-finding tasks', 'Out of class experiences', 'Problem-based Learning sessions', 'Internet search', 'Online course material', 'Practical classes', 'Studying movies, audiofiles, etc', 'Interactive games', 'Case studies', 'Classroom sessions', 'Online participation', 'Movie viewing', 'Practicals', 'Internet research', 'Individual assessment task preparation', 'Team assessment task preparation', 'Case-based inquiry (incl self-directed learning)'}
Unmatched ass_dist keys (26): {'Weekly study questions', 'Interview', '3-minute film', 'Research', 'Practical experiments', 'Field trip', 'Simulation', 'Research proposal', 'Problem-based Learning sessions', 'Instant movies', 'Case analysis', 'Field visit observation notes', 'Infographic diagram', 'Film Festival', 'Data analysis', 'Group activity', 'Problem sets', 'Film production', 'Virtual reality story', 'Ele

/tmp/ipykernel_18050/2036175640.py:273: RuntimeWarning: Mean of empty slice
  df_main["enrol_popularity_pct_sem_adj"] = np.nanmean(np.vstack([s1_pct, s2_pct]), axis=0).round(4)


,course_code,course_name,aoi,is_online,is_comm_intensive,sdg_01,sdg_02,sdg_03,sdg_04,sdg_05,...,llm_perceived_workload,aoi_CCCH,aoi_CCGL,aoi_CCHU,aoi_CCST,sem_both,sem_sem1,sem_sem2,sem_unknown,enrol_popularity_pct_sem_adj
0,CCCH5006,China’s Modernization in the East Asian Contex...,CCCH,1,0,0,0,0,0,0,...,NaN,1,0,0,0,0,1,0,0,NaN
1,CCCH5017,"People, Propaganda and Profit: Understanding M...",CCCH,1,0,0,0,0,0,0,...,2.4,1,0,0,0,0,0,1,0,0.4886
2,CCCH5039,Curing the Chinese: Medicine and Society in Mo...,CCCH,1,0,0,0,1,0,0,...,2.8,1,0,0,0,0,0,1,0,0.8750
3,CCCH5044,"Dunhuang and the Silk Road: Art, Culture and T...",CCCH,1,0,0,0,0,0,0,...,NaN,1,0,0,0,0,0,1,0,NaN
4,CCCH5054,Mothering China: From the Womb to the Nation (...,CCCH,1,0,0,0,1,0,1,...,3.0,1,0,0,0,0,1,0,0,0.8193
